# Prompt Evaluation: Generating Test Datasets
This notebook builds a small AWS-focused evaluation dataset you can reuse to test prompt quality across Python, JSON, and regex tasks.

[Open Lesson Notes](../s04_prompt_evaluation/ss03_generating_test_datasets/index.md)

In [95]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## 1. Initialize Client
Load environment variables and initialize the Anthropic client and model selection used for dataset generation.

In [96]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, stop_sequences=None):
    if stop_sequences is None:
        stop_sequences = []
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

## 2. Define Helper Functions
These helpers standardize message formatting and API calls so the rest of the notebook stays simple and reusable.

In [97]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

## 3. Generate a Small Evaluation Dataset
The function asks Claude to return a compact JSON array of AWS task prompts that are easy to score during evals.
It also includes a fallback parser to extract the JSON array if extra text is returned.

In [98]:
dataset = generate_dataset()

dataset 

[{'task': 'Extract all IAM role ARNs from an AWS CloudTrail log JSON output',
  'format': 'regex'},
 {'task': 'Parse an AWS S3 bucket policy and return a JSON object with allowed principals and actions',
  'format': 'json'},
 {'task': "Write a Python function that converts an AWS region code (e.g., 'us-east-1') to its full region name (e.g., 'US East N. Virginia')",
  'format': 'python'}]

## 4. Persist the Dataset
Save the generated dataset to dataset.json so it can be reused in the next evaluation steps.

In [99]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

## 5. Build the Core Evaluation Functions
Now that the dataset is saved, we can build the evaluation pipeline: run each task, capture the model output, and attach a score plus reasoning.

[Open SS04 Lesson Notes: Running the Eval](../s04_prompt_evaluation/ss04_running_the_eval/index.md)

In [100]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON only. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages)

    # Fallback: extract the first JSON object if extra text is returned.
    start = eval_text.find("{")
    end = eval_text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("Model response did not contain a JSON object.")

    return json.loads(eval_text[start:end + 1])

### grade_by_model
This grader asks Claude to score each solution and return structured JSON with strengths, weaknesses, reasoning, and a numeric score.
It includes fallback JSON extraction so the evaluation can continue even when extra text is returned.

[Open SS05 Lesson Notes: Model Based Grading](../s04_prompt_evaluation/ss05_model_based_grading/index.md)

In [101]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

### run_prompt
This function merges a test case task into the prompt template, sends it to Claude, and returns the raw output.

In [102]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


### Code-Based Syntax Grading
These helper validators check whether model output is valid JSON, Python, or regex syntax and return a binary syntax score (10 or 0).
The `grade_syntax` router selects the correct validator using each test case `format` field.

[Open SS06 Lesson Notes: Code Based Grading](../s04_prompt_evaluation/ss06_code_based_grading/index.md)

In [103]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

### run_test_case
This function runs a single test case, combines model and syntax grading, and returns the output, score, and reasoning.

In [104]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

### run_eval
This function loops through the dataset, calls the single-case runner, and collects all results into one list.

In [105]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

result = run_eval(dataset)

Average score: 6.333333333333333


## 6. Run and Review Results
Load the saved dataset, run the full evaluation, and print formatted JSON to inspect generated outputs, scores, and reasoning.

In [106]:
print(json.dumps(result, indent=2))

[
  {
    "output": "\nimport json\nimport re\nimport sys\n\ndef extract_iam_role_arns(cloudtrail_log):\n    \"\"\"Extract all IAM role ARNs from CloudTrail log\"\"\"\n    \n    if isinstance(cloudtrail_log, str):\n        try:\n            data = json.loads(cloudtrail_log)\n        except json.JSONDecodeError:\n            data = cloudtrail_log\n    else:\n        data = cloudtrail_log\n    \n    role_arns = set()\n    \n    # Pattern for IAM role ARNs\n    role_pattern = r'arn:aws:iam::\\d+:role/[a-zA-Z0-9\\-_/.]+'\n    \n    # Convert data to string to search\n    data_str = json.dumps(data)\n    \n    # Find all matching ARNs\n    matches = re.findall(role_pattern, data_str)\n    role_arns.update(matches)\n    \n    return sorted(list(role_arns))\n\n# Read from stdin if provided\nif __name__ == \"__main__\":\n    try:\n        input_data = sys.stdin.read()\n        result = extract_iam_role_arns(input_data)\n        print(json.dumps(result, indent=2))\n    except Exception as e:\n 